# rustymath

> Maths in plain Rust, from fast f16, f32 and f64 functions to exact rationals.

rustymath gives you maths in plain Rust and in Python, from fast functions on 16-, 32- and 64-bit floats to exact rationals. It covers the elementary functions, such as `exp`, `log`, `sin` and `pow`, at 16, 32 and 64 bits. Most of them also take complex numbers. It also has special functions such as `gamma` and `erf`, probability distributions, reductions and scans, searching and sorting, linear algebra, polynomials, number theory, and APL-inspired tolerant comparison. In both languages, it also computes exactly on rationals: matrix solves and determinants, polynomials, gcds and binomial coefficients.

In Rust, you call each function on single values inside your own loops, or through an object that runs it over a whole slice. In Python, each function takes numbers, lists, rustymath arrays or NumPy arrays, and returns a rustymath array. The Python package doesn't need NumPy.

Its float functions give up the last bits of accuracy for speed. Every elementary and special function is within 4 units in the last place (ulps) of the correctly rounded result. At 32 bits, most functions run two to six times as fast as the standard library's versions.

## Install

Install the Python package from PyPI. For Rust, see [Using rustymath from Rust](#using-rustymath-from-rust).

```sh
pip install rustymath
```

## Using rustymath from Python

Each function takes numbers, nested lists, rustymath arrays, NumPy arrays, or any other object that exports a buffer. It returns a `rustymath.Array`, or a Python number when every argument is a Python number. Arguments of different shapes broadcast by aligning their leading axes, where NumPy aligns trailing axes. An axis of length 1 stretches to match. Here a vector of two items adds one number to each row of a 2 by 3 matrix:

In [ ]:
import rustymath as rm

In [ ]:
rm.add([[1, 2, 3], [4, 5, 6]], [10, 20])

Arrays of different dtypes promote as in NumPy: the result takes the higher kind, wide enough to hold both, so an `i16` array beside an `f16` array gives `f32`. Characters don't mix with numbers. An integer result that doesn't fit its dtype raises `OverflowError`. A Python number or list follows NEP 50, as in NumPy: it takes the dtype of the array beside it, unless its kind is higher, so a float beside an integer array gives `f64`. A function that computes only on floats, such as `exp`, converts Booleans and integers to the narrowest float that holds them exactly, but at least `f32`, which computes faster than `f16`. So Booleans, `u8` and `i16` give `f32`, and wider integers give `f64`:

In [ ]:
i16 = rm.array([1, 2]).astype('i16')
rm.add(i16, 3), rm.add(i16, 0.5), rm.add(i16, rm.array([0.5, 1]).astype('f16')), rm.exp(i16), rm.exp([1, 2])

A `rustymath.Array` has arithmetic and comparison operators, `@`, and the methods `.sum()`, `.prod()`, `.max()`, `.min()`, `.all()`, `.any()`, `.cumsum()`, `.cumprod()`, `.reshape()`, `.astype()` and `.tolist()`. With no `axis`, a reduction reduces the whole array to a number, and a scan runs over the flattened array:

In [ ]:
m = rm.array([[1, 2], [3, 4]])
(m * 10).sum(), m.sum(axis=0), m.cumsum()

A dictionary becomes a vector whose axis has the dictionary's keys. Functions align keyed axes by key:

In [ ]:
rm.add({'a': 1.5, 'b': 2.5}, {'b': 10.0, 'a': 20.0})

Assigning a tuple to `.names` names the axes, with one name or `None` for each axis. `axis` then takes a name wherever it takes a position:

In [ ]:
s = rm.array([[10, 20, 30], [40, 50, 60]])
s.names = ('city', 'month')
s.sum(axis='month')

The rest of this page also uses NumPy:

In [ ]:
import numpy as np

Where NumPy has a function, rustymath uses the same name and arguments. NumPy functions take rustymath arrays directly. `np.asarray` views a rustymath array as a NumPy array, with no copy. Here `exp` keeps the 32-bit width of its argument:

In [ ]:
x = np.linspace(0, 2, 5, dtype=np.float32)
rm.exp(x)

array([1.       , 1.6487211, 2.718282 , 4.4816885, 7.389056 ],
      dtype=float32)

To check the accuracy yourself, compare with NumPy. `np.spacing` gives the size of one ulp at each value. Dividing the difference by it gives the distance between the two results in ulps:

In [ ]:
xs = np.linspace(-10, 10, 100_001)
y = np.exp(xs)
(np.abs(rm.exp(xs) - y) / np.spacing(y)).max()

np.float64(2.0)

Functions that NumPy lacks keep their own names. `root(n, y)` gives the `n`th root of `y`:

In [ ]:
rm.root(3, np.array([8., 27., 1000.]))

array([ 2.,  3., 10.])

The Rust names follow Rust's standard library instead. Python's `log`, `power`, `arcsin` and `arctan2` are Rust's `ln`, `pow`, `asin` and `atan2`.

### Complex numbers

A complex array gives complex results, at each function's principal value. For `exp`, `log`, `power`, `sin`, `cos`, `sinh` and `cosh`, rustymath computes a complex array through its vectorized real loops:

In [ ]:
rm.sqrt(np.array([-4+0j, 3+4j])), rm.log(np.array([-1+0j]))

(array([0.+2.j, 2.+1.j]), array([0.+3.14159265j]))

### Special functions and distributions

The special functions take SciPy's names, such as `gamma`, `gammaln`, `erf`, `erfinv`, `betainc` and `lambertw`:

In [ ]:
rm.gamma(np.array([0.5, 5, 10])), rm.erfinv(0.5)

(array([1.77245385e+00, 2.40000000e+01, 3.62880000e+05]),
 np.float64(0.4769362762044699))

`rm.distribution` holds sixteen distributions. Each constructor takes the distribution's parameters, or uses standard ones where the distribution has them. The result has `pdf`, `cdf`, `quantile` and `sample`. The same `seed` gives the same draws:

In [ ]:
n = rm.distribution.normal()
n.cdf([-1.96, 0, 1.96]), n.quantile(0.975), n.sample(3, seed=0)

(array([0.0249979, 0.5      , 0.9750021]),
 1.9599639845400538,
 array([ 0.11623959, -0.78285485, -2.33463531]))

### Polynomials

`rm.polynomial` evaluates polynomials, finds roots, constructs coefficients from roots, differentiates and fits data, under the names of `numpy.polynomial.polynomial`. Coefficients start from the constant term. Evaluation, construction and derivatives take real or complex arrays. `polyvalfromroots(x, roots, m=1)` evaluates directly from roots without forming coefficients. The roots of x^3 - 6x^2 + 11x - 6 are 1, 2 and 3:

In [ ]:
rm.polynomial.polyroots([-6, 11, -6, 1])

array([1.+0.j, 2.-0.j, 3.+0.j])

### Linear algebra

`rm.linalg` provides real and complex matrix solves and decompositions, including `solve`, `det`, `lu`, `cholesky`, `qr`, `svd`, `eigh`, `eig`, `lstsq`, `pinv` and `matrix_rank`. They take stacks of matrices, whose leading axes broadcast from the front as other functions' arguments do. `solve` and `pinv` give least-norm least-squares solutions for singular or non-square matrices. `permanent` takes square matrices. `echelon` reduces one 2-D matrix. `lu` gives the packed factors and the row positions. Its `.l` and `.u` give the separate triangular matrices. Reuse the factors for solves, inverses and determinants:

In [ ]:
a = np.array([[4., 1], [1, 3]])
f = rm.linalg.lu(a)
f.solve([1., 2]), f.determinant(), rm.linalg.eigh(a).values, rm.linalg.solve([[1., 2], [2, 4]], [1., 2])

(array([0.09090909, 0.63636364]),
 np.float64(11.0),
 array([2.38196601, 4.61803399]),
 array([0.2, 0.4]))

`rm.matmul` and the `@` operator multiply matrices. Integer matrices multiply in `i64`. A product that doesn't fit raises `OverflowError`. Products and solves pair keyed axes by key. Their results keep the keys of the axes they index. `rm.linalg.semiring(f, g)` gives the matrix product whose sums use `f` and whose products use `g`, as APL's inner product `f.g` does. With `add` and `equal`, it counts the matching items. Squaring a matrix of edge lengths with `minimum` and `add` gives the shortest paths of at most two edges:

In [ ]:
d = np.array([[0, 3, np.inf], [np.inf, 0, 1], [2, np.inf, 0]])
minplus = rm.linalg.semiring(rm.minimum, rm.add)
minplus(d, d)

array([[0., 3., 4.],
       [3., 0., 1.],
       [2., 5., 0.]])

### Optimization

`rm.optimize` takes SciPy's names. `minimize` runs Brent's method, Nelder–Mead, Powell's method, L-BFGS or differential evolution. Without a `method`, it picks one from the number of variables, whether `jac` gives a gradient and whether there are bounds. `root_scalar` finds a zero of a function of one variable by Brent's method, within a bracket or by stepping outward from a start. Here Nelder–Mead finds the minimum of Rosenbrock's function, at (1, 1), and Brent's method finds where cos x = x:

In [ ]:
rosen = lambda x: (1-x[0])**2 + 100*(x[1]-x[0]**2)**2
rm.optimize.minimize(rosen, [-1.2, 1]).x, rm.optimize.root_scalar(lambda x: np.cos(x) - x, [0, 1]).root

(array([1., 1.]), 0.7390851350713885)

`least_squares` fits by Levenberg–Marquardt. `cg`, `minres`, `gmres` and `lsmr` solve linear systems through the matrix's products alone. They take the matrix as an array or as anything with a `shape` and a `matvec` method, such as a SciPy `LinearOperator`. An array's products run in Rust, with no Python call for each product. Here `least_squares` recovers a = 2 and b = 0.3 from five points on y = a e^(bx), and `lsmr` gives the least-squares line through five points that no line fits:

In [ ]:
t = np.linspace(0, 2, 5)
fit = rm.optimize.least_squares(lambda p: p[0]*np.exp(p[1]*t) - 2*np.exp(0.3*t), [1, 1])
line = rm.optimize.lsmr(np.column_stack([np.ones(5), np.arange(5.)]), [1, 2.9, 5.2, 6.8, 9.1])
fit.x, line.x

### Number theory

`rm.number_theory` tests primality, factorises, finds the next or previous prime, gives Euler's totient and exact factorials, and lists and counts primes, on Python integers of any size. Ctrl-C stops a long call:

In [ ]:
nt = rm.number_theory
nt.factor(2**64 + 1), nt.prime_count(10**8), nt.next_prime(10**20)

([(274177, 1), (67280421310721, 1)], 5761455, 100000000000000000039)

### Exact arithmetic

`rm.exact` takes integers and `fractions.Fraction`. It provides rational gcd, lcm, remainder and binomial coefficients, matrix solves, pseudoinverses, determinants, permanents and row reduction, and polynomial construction, evaluation and derivatives. Arguments are integers, Fractions, nested lists of them, or arrays of integers. Results are Fractions, in nested lists for arrays. Matrix calls take one matrix. Ordinary `rm.linalg` calls stay floating-point.

In [ ]:
from fractions import Fraction

In [ ]:
rm.exact.solve([[2, 1], [1, 2]], [Fraction(1, 2), 0]), rm.exact.binom(Fraction(3, 2), 2)

(array([Fraction(1, 3), Fraction(-1, 6)], dtype=object), Fraction(3, 8))

### Tolerant comparison

APL compares floats within a relative tolerance. Two floats are equal when their difference is less than the tolerance times the larger magnitude. With the usual tolerance, `0.1 + 0.2` equals `0.3`. `rm.equal`, `not_equal`, `less`, `less_equal`, `greater` and `greater_equal` compare this way. Their tolerance `ct` defaults to `rm.CT`, 1e-14. Pass `ct=0` to compare exactly:

In [ ]:
rm.equal(0.1 + 0.2, 0.3), rm.equal(0.1 + 0.2, 0.3, ct=0)

(np.True_, np.False_)

`floor`, `ceil`, `remainder`, `gcd` and `lcm` take `ct` too. A value within tolerance of a whole number counts as that whole number:

In [ ]:
rm.floor(np.array([2.9999999999999996, 2.5]))

array([3., 2.])

`remainder` takes its arguments in NumPy's order. `rm.remainder(x, y)` is the Rust `residue(y, x)`.

`unique`, `index` and `isin` search floats within the same tolerance. They match Booleans and integers exactly. `unique` gives the values in order of first occurrence, with each one's first position, each item's class number and each class's size. `index(x, y)` gives where in `x` each item of `y` first matches, or the size of `x` where nothing does:

In [ ]:
rm.unique([0.3, 1.0, 0.1 + 0.2]), rm.index([0.3, 1.0], 0.1 + 0.2)

(Unique(values=array([0.3, 1. ]), firsts=array([0, 1]), numbers=array([0, 1, 0]), counts=array([2, 1])),
 np.int64(0))

`rm.search` has stable `argsort` and `lexsort`, `searchsorted`, integer `bincount`, class `counts` and `group_indices`. These functions are also available directly on `rm`. `argsort` and `lexsort` sort along the last axis unless you give `axis`, as NumPy's sorts do. Sorting and insertion searches compare exactly and put NaNs last. `rm.integer.encode` and `decode` convert mixed-radix integers in `i64`. A result that doesn't fit raises `OverflowError`. `rm.polynomial.decode` and `rm.exact.decode` accept field-valued bases and digits.

`rm.minmax(x)` gives the least and greatest items together. `rm.all` and `rm.any` reduce Booleans. `rm.logical_scan(x, table)` scans any Boolean dyad given by its 2×2 truth table. `rm.set_threads(n)` sets the calling thread's limit and returns its previous setting. Use 0 for the default or 1 for serial calls.

{{< include ../src/lib.md >}}

## Licence

Apache-2.0. Some algorithms and constants come from SLEEF and from Arm Optimized Routines. `THIRD-PARTY.md` has their notices.